# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements such as record sets, fields, and columns are referenced by their Croissant `@id` values.

### Dataset Source

The dataset is provided by a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Install mlcroissant if not already installed
!pip install -U mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset metadata and create a Dataset object
dataset = mlc.Dataset(croissant_url)

# Print dataset-level metadata
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Publication Date: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, columns and their `@id`s according to the Croissant schema.

In [ ]:
# List all record sets, their fields and columns by @id using the Croissant metadata
recordsets = list(dataset.metadata.recordSets)

if len(recordsets) == 0:
    print("No record sets found in the dataset metadata.")
else:
    print(f"Found {len(recordsets)} record sets:")
    for record_set in recordsets:
        print(f"- RecordSet @id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', '(unnamed)')}")
        # List fields
        fields = record_set.get('fields', [])
        print(f"  Fields:")
        for field in fields:
            print(f"    - Field @id: {field['@id']}")
            print(f"      Name: {field.get('name', '(unnamed)')}")
            if 'dataType' in field:
                print(f"      DataType: {field['dataType']}")
            if 'column' in field:
                print(f"      Source column(s): {[c['@id'] for c in (field['column'] if isinstance(field['column'], list) else [field['column']]) ]}")
        print()
    
    # Show a sample of one record set's first 2 records
    sample_record_set_id = recordsets[0]['@id']
    print(f"Sample records from RecordSet @id={sample_record_set_id}:")
    for i, row in enumerate(dataset.records(record_set=sample_record_set_id)):
        if i >= 2:
            break
        print(row)

## 3. Data Extraction
Load data from all available record sets into DataFrames for analysis. Each data frame is indexed by its record set `@id`.

**Note**: All entities are referenced via their `@id` field.

In [ ]:
# Collect all record set @id values
record_set_ids = [rs['@id'] for rs in recordsets]
dataframes = dict()
for recset_id in record_set_ids:
    rec_list = list(dataset.records(record_set=recset_id))
    df = pd.DataFrame(rec_list)
    dataframes[recset_id] = df
    print(f"Loaded {len(df)} records from RecordSet: {recset_id}")

# Display columns of the primary record set and preview contents (use the first recordset for demo)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nColumns in main record set (@id={main_record_set_id}):")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
This section demonstrates filtering, normalization, grouping, and summary statistics. All fields are referenced using their `@id` as given in the previous metadata overview.

Modify the field and record set `@id`s below as appropriate for your analysis. For demonstration, the code attempts to auto-detect some likely numeric fields.

In [ ]:
# Choose a numeric field for analysis by inspecting column types
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Try to auto-detect a numeric field
    numeric_field_id = None
    for col in df.columns:
        # Heuristic: look for integer or float columns
        if np.issubdtype(df[col].dropna().astype(str).str.replace('.', '', 1).str.isdigit().astype(bool).dtype, np.bool_):
            try:
                vals = pd.to_numeric(df[col], errors='coerce')
                if vals.notnull().all():
                    numeric_field_id = col
                    break
            except Exception:
                continue
    # For demonstration purposes, fallback to the first column if not found
    if numeric_field_id is None and len(df.columns)>0:
        numeric_field_id = df.columns[0]

    print(f"Analyzing field '@id' = {numeric_field_id}")
    # Convert to numeric, drop NA for EDA
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

    # Only analyze if there's at least some non-null numeric data
    if df[numeric_field_id].notnull().sum() >= 1:
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try to auto-detect a categorical/group field
        group_field_id = None
        for col in df.columns:
            # exclude current field and look for object columns with few unique values
            if col != numeric_field_id and df[col].dtype == 'object' and df[col].nunique() < len(df)//4:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field '@id' = {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No suitable group field detected for grouping.")

    else:
        print('No valid numeric data detected for analysis.')
else:
    print('Main record set not loaded; cannot proceed with EDA.')

## 5. Visualization

Visualize field distributions or relationships. Below we plot a histogram for the chosen numeric field if available. Entities are referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True, color='cornflowerblue')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If a group_field_id exists, visualize grouped mean as a barplot
    if 'group_field_id' in locals() and group_field_id:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().dropna()
        if group_means.shape[0]>1:
            group_means.plot(kind='bar', color='teal', figsize=(8,4))
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.xlabel(group_field_id)
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.show()

## 6. Conclusion

- This notebook demonstrated loading and exploration of the Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors dataset using the `mlcroissant` library.
- All entities (record sets, fields, columns) were referenced via their Croissant `@id`.
- We loaded data into pandas DataFrames, performed basic EDA and normalization, and visualized selected variables.
- For further analyses, refer to the schema and field ids in the Croissant metadata for precise entity references.
